In [ ]:
import papermill as pm
from pathlib import Path
import os
from itertools import product


def run_notebook(notebook_to_run, parameter_dict):
    # remember old working directory
    old_wd = os.getcwd()
    try:
        # change to directory of notebook before running
        # NOTE: this may be necessary to ensure relative imports get resolved
        os.chdir(Path(notebook_to_run).absolute().parent)

        # tell recipe_params which notebook is being run and how. NOTE: the kernel
        # that papermill starts inherits the environment of this process, so the
        # variables below are visible inside the recipe notebook (see recipe_params.py)
        os.environ["RECIPE_NOTEBOOK"] = str(notebook_to_run)
        os.environ["RECIPE_RUNNER"] = "papermill"

        # run notebook, overwrite parameters in parameter_dict
        # NOTE: papermill saves a copy of the modified notebook by default, we pass it to /dev/null to ignore
        # NOTE: the values that were actually used are dumped to parameter_runs/ by the
        #       "parameter-capture" cell that every recipe has (scripts/inject_parameter_logging.py)
        pm.execute_notebook(Path(notebook_to_run).name, '/dev/null', parameters=parameter_dict)

    # change back to old working directory
    finally:
        os.chdir(old_wd)


## 1. Setup work list

Define a list of parameter dictionaries with which the notebook should be run (see below for alternatives)

In [ ]:
parameter_dicts = [
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25AM11-01_3'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25AM11-01_5'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25AM11-03_4'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25AM11-03_6'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25AM11-04_4'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25AM12-02_1'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25AM12-02_3'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25JV_006'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/25JV_008'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/26AM01-01_2'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/26AM01-01_4'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/26JV_018'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/26JV_020'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/26JV_022'},
    # {'in_path': '/Volumes/nn/Julia Vogtmann/Microscopy/26JV_024'},
    {'in_path': '/home/david/Desktop/symbiosis_analysis_guse/'}
]

# extra parameter adjustments for each run
# specify as list of dicts, if multiple are given we form the cartesian product with the list of dicts above
# e.g. you might want to run edge-snap with multiple segmentations of the same dataset
extra_params = [
    # {'mask_subdirectory': 'segmentation_nucleoli', 'out_subdirectory': 'segmentation_nucleoli_edgesnap', 'mask_suffix': '_segmented'},
    # {'mask_subdirectory': 'segmentation_nuclei', 'out_subdirectory': 'segmentation_nuclei_edgesnap', 'mask_suffix': '_cp_masks'}
]

if len(extra_params) > 0:
    parameter_dicts = [p1 | p2 for (p1, p2) in product(parameter_dicts, extra_params)]

parameter_dicts

## 2. Run Notebook

In [ ]:
notebook_to_run = 'resave/resave_nd2_as_tiff.ipynb'

for parameter_dict in parameter_dicts:
    print(f'running {notebook_to_run} with parameters {parameter_dict}.')
    run_notebook(notebook_to_run, parameter_dict)

### 1a. Alternative: Process all subdirectories

In [ ]:
import os

# get all subdirectories of a base dir
basedir, dirs, _ = next(os.walk('/Volumes/nn/Julia Vogtmann/Microscopy'))

# make worklist
parameter_dicts = [{'in_path': os.path.join(basedir, d)} for d in sorted(dirs)]
parameter_dicts

### 3. What was run?

The recipe notebooks log their effective parameter values to `parameter_runs/` (`<notebook>.json` for the latest run, `runs.jsonl` as an append-only log). Summarise the runs that were just executed:

In [ ]:
import json
import recipe_params

log_path = recipe_params.repo_root() / 'parameter_runs' / 'runs.jsonl'
runs = [json.loads(l) for l in log_path.read_text().splitlines()] if log_path.exists() else []
runs = [r for r in runs if r['notebook'] == notebook_to_run][-len(parameter_dicts):]

print(f'{len(runs)} of {len(parameter_dicts)} run(s) recorded for {notebook_to_run}')
for r in runs:
    overrides = r['changed'] or '(defaults from the file)'
    print(f"  {r['timestamp']}  in_path={r['parameters'].get('in_path')!r}  overrides={overrides}".rstrip())
